In [13]:
import requests
import pandas as pd
import time
from pathlib import Path

# API adresi
url = "https://earthquake.usgs.gov/fdsnws/event/1/query"

# Yıllık CSV dosyalarının kaydedileceği klasör
klasor = Path("deprem_csv")
klasor.mkdir(exist_ok=True)

# Veri çekilecek tarih aralığı
baslangic_yili = 2000
bitis_yili = pd.Timestamp.now(tz="UTC").year

# API bağlantısı oluştur
session = requests.Session()

# Her yılın verilerini indir veya mevcut dosyayı koru
for yil in range(baslangic_yili, bitis_yili + 1):

    dosya = klasor / f"deprem_{yil}.csv"

    # Daha önce kaydedilmiş yılları tekrar indirme
    if dosya.exists():
        print(f"{yil}: Dosya zaten mevcut, atlanıyor.")
        continue

    params = {
        "format": "geojson",
        "starttime": f"{yil}-01-01",
        "endtime": f"{yil + 1}-01-01",
        "minmagnitude": 4.5,
        "orderby": "time-asc",
        "limit": 20000
    }

    # Başarısız istekleri yeniden dene
    for deneme in range(1, 6):
        try:
            response = session.get(
                url,
                params=params,
                timeout=120
            )
            response.raise_for_status()
            kayitlar = response.json()["features"]
            break

        except (requests.exceptions.RequestException, ValueError) as hata:
            print(
                f"{yil}: {deneme}/5 deneme başarısız. Hata: {hata}"
            )

            if deneme == 5:
                raise RuntimeError(
                    f"{yil} yılı indirilemedi. "
                    "Bağlantıyı kontrol edip kodu tekrar çalıştır."
                ) from hata

            time.sleep(5 * deneme)

    # API yanıtını tabloya dönüştür
    yillik_depremler = []

    for deprem in kayitlar:
        ozellikler = deprem["properties"]
        koordinatlar = deprem["geometry"]["coordinates"]

        yillik_depremler.append({
            "earthquake_id": deprem["id"],
            "date": pd.to_datetime(
                ozellikler["time"],
                unit="ms",
                utc=True
            ),
            "magnitude": ozellikler["mag"],
            "place": ozellikler["place"],
            "longitude": koordinatlar[0],
            "latitude": koordinatlar[1],
            "depth_km": koordinatlar[2]
        })

    # İndirilen yılı CSV olarak kaydet
    df_yil = pd.DataFrame(yillik_depremler)

    df_yil.to_csv(
        dosya,
        index=False,
        encoding="utf-8-sig"
    )

    print(f"{yil}: {len(df_yil)} kayıt kaydedildi.")

    time.sleep(0.5)

session.close()

# Tüm yılların dosyalarını kontrol et
dosyalar = sorted(klasor.glob("deprem_*.csv"))

eksik_yillar = [
    yil
    for yil in range(baslangic_yili, bitis_yili + 1)
    if not (klasor / f"deprem_{yil}.csv").exists()
]

if eksik_yillar:
    print("\nEksik yıllar:", eksik_yillar)
    print("Eksik yıllar tamamlanmadan ana CSV oluşturulmadı.")

else:
    # Yıllık dosyaları birleştir
    df = pd.concat(
        [pd.read_csv(dosya) for dosya in dosyalar],
        ignore_index=True
    )

    # Yinelenen deprem kayıtlarını kaldır
    df = df.drop_duplicates(subset="earthquake_id")

    # Farklı biçimlerdeki tarihleri birlikte oku
    df["date"] = pd.to_datetime(
        df["date"],
        format="mixed",
        utc=True
    )

    # Tarihe göre sırala
    df = df.sort_values("date").reset_index(drop=True)

    # Birleştirilmiş CSV dosyasını kaydet
    df.to_csv(
        "deprem_verileri.csv",
        index=False,
        encoding="utf-8-sig"
    )

    # Sonuçları kontrol et
    print("\n--- VERİ TOPLAMA TAMAMLANDI ---")
    print("Toplam kayıt:", len(df))
    print("Sütun sayısı:", len(df.columns))
    print("İlk tarih:", df["date"].min())
    print("Son tarih:", df["date"].max())
    print("Eksik değerler:\n", df.isna().sum())
    print("\nİlk 5 kayıt:")
    display(df.head())

2000: 5010 kayıt kaydedildi.
2001: 4680 kayıt kaydedildi.
2002: 4963 kayıt kaydedildi.
2003: 5136 kayıt kaydedildi.
2004: 6183 kayıt kaydedildi.
2005: 7878 kayıt kaydedildi.
2006: 7075 kayıt kaydedildi.
2007: 7274 kayıt kaydedildi.
2008: 6307 kayıt kaydedildi.
2009: 6054 kayıt kaydedildi.
2010: 7214 kayıt kaydedildi.
2011: 9584 kayıt kaydedildi.
2012: 7189 kayıt kaydedildi.
2013: 7211 kayıt kaydedildi.
2014: 8333 kayıt kaydedildi.
2015: 7153 kayıt kaydedildi.
2016: 7440 kayıt kaydedildi.
2017: 6372 kayıt kaydedildi.
2018: 7571 kayıt kaydedildi.
2019: 7211 kayıt kaydedildi.
2020: 6511 kayıt kaydedildi.
2021: 8959 kayıt kaydedildi.
2022: 7766 kayıt kaydedildi.
2023: 7651 kayıt kaydedildi.
2024: 6397 kayıt kaydedildi.
2025: 8560 kayıt kaydedildi.
2026: 6510 kayıt kaydedildi.

--- VERİ TOPLAMA TAMAMLANDI ---
Toplam kayıt: 188192
Sütun sayısı: 7
İlk tarih: 2000-01-01 01:19:26.990000+00:00
Son tarih: 2026-10-27 15:31:42.073000+00:00
Eksik değerler:
 earthquake_id    0
date             0
magn

,earthquake_id,date,magnitude,place,longitude,latitude,depth_km
0,usp0009kjw,2000-01-01 01:19:26.990000+00:00,4.5,"14 km SW of Dragash, Kosovo",20.543,41.927,10.0
1,usp0009kk5,2000-01-01 04:30:10.370000+00:00,4.5,"130 km ENE of Diego de Almagro, Chile",-68.803,-26.012,105.0
2,usp0009kk8,2000-01-01 05:24:35.290000+00:00,5.1,"29 km SSE of Rust?q, Afghanistan",69.947,36.874,54.3
3,usp0009kkb,2000-01-01 05:58:19.780000+00:00,6.0,west of Macquarie Island,153.670,-60.722,10.0
4,usp0009kkh,2000-01-01 07:50:00.390000+00:00,4.5,"0 km SE of Hinatuan, Philippines",126.336,8.370,100.7


In [14]:
# Veri kalitesi kontrolleri
print("Satır ve sütun sayısı:", df.shape)
print("Tekrarlanan deprem ID:", df["earthquake_id"].duplicated().sum())
print("Eksik değerler:\n", df.isnull().sum())
print("Tarih aralığı:", df["date"].min(), "-", df["date"].max())
print("Büyüklük aralığı:", df["magnitude"].min(), "-", df["magnitude"].max())
print("Geçersiz koordinat sayısı:", (
    ~df["latitude"].between(-90, 90) |
    ~df["longitude"].between(-180, 180)
).sum())


Satır ve sütun sayısı: (188192, 7)
Tekrarlanan deprem ID: 0
Eksik değerler:
 earthquake_id    0
date             0
magnitude        0
place            0
longitude        0
latitude         0
depth_km         0
dtype: int64
Tarih aralığı: 2000-01-01 01:19:26.990000+00:00 - 2026-10-27 15:31:42.073000+00:00
Büyüklük aralığı: 3.38 - 9.1
Geçersiz koordinat sayısı: 0


In [15]:
# Orijinal veriyi koruyarak temiz bir kopya oluştur
df_clean = df.copy()

# Veri türlerini standartlaştır
df_clean["date"] = pd.to_datetime(df_clean["date"], utc=True, errors="coerce")
numeric_cols = ["magnitude", "longitude", "latitude", "depth_km"]

for col in numeric_cols:
    df_clean[col] = pd.to_numeric(df_clean[col], errors="coerce")

# Zorunlu alanları eksik kayıtları kaldır
df_clean = df_clean.dropna(
    subset=["earthquake_id", "date", "magnitude",
            "longitude", "latitude", "depth_km"]
)

# Tekrarlanan deprem kayıtlarını kaldır
df_clean = df_clean.drop_duplicates(subset="earthquake_id")

# Geçersiz koordinatları kaldır
df_clean = df_clean[
    df_clean["latitude"].between(-90, 90) &
    df_clean["longitude"].between(-180, 180)
]

# Projenin belirlediği minimum büyüklük koşulunu uygula
df_clean = df_clean[df_clean["magnitude"] >= 4.5]

# 2000 yılından bugüne kadar olan kayıtları tut
simdi = pd.Timestamp.now(tz="UTC")
df_clean = df_clean[
    (df_clean["date"] >= "2000-01-01") &
    (df_clean["date"] <= simdi)
]

# İndeksi düzenle
df_clean = df_clean.sort_values("date").reset_index(drop=True)

# Temizlik sonuçlarını kontrol et
print("İlk kayıt sayısı:", len(df))
print("Temizlenen veri seti:", len(df_clean))
print("Silinen kayıt sayısı:", len(df) - len(df_clean))
print("Minimum büyüklük:", df_clean["magnitude"].min())
print("Eksik değer sayısı:", df_clean.isnull().sum().sum())
print("Tekrarlanan ID:", df_clean["earthquake_id"].duplicated().sum())

display(df_clean.head())


İlk kayıt sayısı: 188192
Temizlenen veri seti: 188191
Silinen kayıt sayısı: 1
Minimum büyüklük: 4.5
Eksik değer sayısı: 0
Tekrarlanan ID: 0


,earthquake_id,date,magnitude,place,longitude,latitude,depth_km
0,usp0009kjw,2000-01-01 01:19:26.990000+00:00,4.5,"14 km SW of Dragash, Kosovo",20.543,41.927,10.0
1,usp0009kk5,2000-01-01 04:30:10.370000+00:00,4.5,"130 km ENE of Diego de Almagro, Chile",-68.803,-26.012,105.0
2,usp0009kk8,2000-01-01 05:24:35.290000+00:00,5.1,"29 km SSE of Rust?q, Afghanistan",69.947,36.874,54.3
3,usp0009kkb,2000-01-01 05:58:19.780000+00:00,6.0,west of Macquarie Island,153.670,-60.722,10.0
4,usp0009kkh,2000-01-01 07:50:00.390000+00:00,4.5,"0 km SE of Hinatuan, Philippines",126.336,8.370,100.7


In [27]:
# Temizlenmiş verileri CSV dosyasına kaydet
df_clean.to_csv(
    "deprem_verileri.csv",
    index=False,
    encoding="utf-8-sig"
)

In [26]:
df[['magnitude', 'longitude', 'latitude', 'depth_km']].head()
df.dtypes

earthquake_id            object
date             datetime64[ns]
magnitude               float64
place                    object
longitude               float64
latitude                float64
depth_km                float64
dtype: object

In [ ]:
import pandas as pd
import os
from dotenv import load_dotenv
from sqlalchemy import create_engine

load_dotenv()
engine = create_engine(
    f"mssql+pyodbc://@{os.getenv('SQL_SERVER')}/{os.getenv('SQL_DB')}"
    "?driver=ODBC+Driver+17+for+SQL+Server&trusted_connection=yes"
)

df_sql = pd.read_sql("SELECT * FROM deprem_verileri", engine)
print(len(df_sql))
print(df_sql['date'].min(), df_sql['date'].max())
print(df_sql['magnitude'].min(), df_sql['magnitude'].max())
print(df_sql['depth_km'].isna().sum())

188192
2000-01-01 01:19:26.990000 2026-10-27 15:31:42.073000
3.38 9.1
0


In [31]:
from sqlalchemy import text

with engine.begin() as conn:
    conn.execute(text("IF COL_LENGTH('deprem_verileri','region') IS NULL ALTER TABLE deprem_verileri ADD region NVARCHAR(200)"))
    conn.execute(text("""
        UPDATE deprem_verileri
        SET region = CASE WHEN CHARINDEX(',', place) > 0
            THEN LTRIM(RTRIM(RIGHT(place, CHARINDEX(',', REVERSE(place)) - 1)))
            ELSE place END
    """))

print(pd.read_sql("SELECT TOP 10 region, COUNT(*) AS n FROM deprem_verileri GROUP BY region ORDER BY n DESC", engine))

                          region      n
0                      Indonesia  22822
1                          Japan  12952
2               Papua New Guinea  10844
3                    Philippines   9489
4                          Tonga   8062
5                         Russia   6696
6                          Chile   6435
7                        Vanuatu   5682
8  South Sandwich Islands region   5251
9                Solomon Islands   4457


In [ ]:
from asistan import sor
sql, df, cevap = sor("2023'te Japonya'da 6'dan büyük kaç deprem oldu?")
print(sql)
print(cevap)

SELECT COUNT(*) AS deprem_sayisi
FROM deprem_verileri
WHERE YEAR(date) = 2023
  AND region LIKE '%Japan%'
  AND magnitude > 6
2023 yılında Japonya'da 6'dan büyük 7 deprem gerçekleşmiştir.
